In [18]:
import imaplib
import email
from email.header import decode_header

In [19]:
IMAP_SERVER = "imap.gmail.com"
EMAIL_ACCOUNT = "yuvabedevelopers@gmail.com"
PASSWORD = "yqew xpph qufd oqib"

In [20]:
def fetch_unread_emails():
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(EMAIL_ACCOUNT, PASSWORD)
    mail.select("inbox")

    _, message_numbers = mail.search(None, "UNSEEN")

    emails = []
    for num in message_numbers[0].split():
        _, msg_data = mail.fetch(num, "(RFC822)")
        raw = email.message_from_bytes(msg_data[0][1])

        subject, encoding = decode_header(raw["Subject"])[0]
        if isinstance(subject, bytes):
            subject = subject.decode(encoding or "utf-8")

        from_ = raw.get("From")
        body = ""
        if raw.is_multipart():
            for part in raw.walk():
                if part.get_content_type() == "text/plain":
                    body = part.get_payload(decode=True).decode(errors="ignore")
                    break
        else:
            body = raw.get_payload(decode=True).decode(errors="ignore")

        emails.append({"from": from_, "subject": subject, "body": body, "num": num})
        mail.store(num, "+FLAGS", "\\Seen")
        print(f"📩 From: {from_}\n   Subject: {subject}\n")

    mail.logout()
    return emails

unread = fetch_unread_emails()
print(f"\n✅ Fetched {len(unread)} unread email(s).")

📩 From: Google <no-reply@accounts.google.com>
   Subject: Security alert

📩 From: Google <no-reply@accounts.google.com>
   Subject: 2-Step Verification turned on

📩 From: Google <no-reply@accounts.google.com>
   Subject: Security alert


✅ Fetched 3 unread email(s).


In [21]:
from datetime import datetime

def fetch_emails_by_date(date_str):
    """date_str format: 'DD-Mon-YYYY'  e.g. '05-May-2026'"""
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(EMAIL_ACCOUNT, PASSWORD)
    mail.select("inbox")

    _, message_numbers = mail.search(None, f'ON "{date_str}"')

    emails = []
    for num in message_numbers[0].split():
        _, msg_data = mail.fetch(num, "(RFC822)")
        raw = email.message_from_bytes(msg_data[0][1])

        subject, encoding = decode_header(raw["Subject"])[0]
        if isinstance(subject, bytes):
            subject = subject.decode(encoding or "utf-8")

        from_ = raw.get("From")
        date = raw.get("Date")

        body = ""
        if raw.is_multipart():
            for part in raw.walk():
                if part.get_content_type() == "text/plain":
                    body = part.get_payload(decode=True).decode(errors="ignore")
                    break
        else:
            body = raw.get_payload(decode=True).decode(errors="ignore")

        emails.append({"from": from_, "subject": subject, "date": date, "body": body})

    mail.logout()
    return emails


TARGET_DATE = "05-May-2026"  # change this
results = fetch_emails_by_date(TARGET_DATE)

print(f"📬 {len(results)} email(s) on {TARGET_DATE}\n" + "─" * 50)
for i, e in enumerate(results, 1):
    print(f"\n[{i}] From    : {e['from']}")
    print(f"    Date    : {e['date']}")
    print(f"    Subject : {e['subject']}")
    print(f"    Body    :\n{e['body'][:300]}{'...' if len(e['body']) > 300 else ''}")
    print("─" * 50)

📬 3 email(s) on 05-May-2026
──────────────────────────────────────────────────

[1] From    : Google <no-reply@accounts.google.com>
    Date    : Tue, 05 May 2026 05:43:28 GMT
    Subject : Security alert
    Body    :
[image: Google]
Recovery phone was changed


yuvabedevelopers@gmail.com
The recovery phone for your account was changed. If you didn't change it,
you should check what happened.
Check activity
<https://accounts.google.com/AccountChooser?Email=yuvabedevelopers@gmail.com&continue=https://myacc...
──────────────────────────────────────────────────

[2] From    : Google <no-reply@accounts.google.com>
    Date    : Tue, 05 May 2026 05:46:50 GMT
    Subject : 2-Step Verification turned on
    Body    :
[image: Google]
2-Step Verification turned on


yuvabedevelopers@gmail.com

Your Google Account yuvabedevelopers@gmail.com is now protected with 2-Step
Verification. When you sign in on a new or untrusted device, you’ll need
your second factor to verify your identity.

*Don't ge

In [ ]:
# TODO: implement webhook trigger
# Will POST each email in `unread` to the webhook endpoint once the API is ready.

WEBHOOK_URL = "https://your-api.com/webhook"  # replace when API is ready

def trigger_webhook(emails):
    import requests
    for e in emails:
        resp = requests.post(WEBHOOK_URL, json={"from": e["from"], "subject": e["subject"]})
        print(f"Sent: {e['subject']} → {resp.status_code}")

# trigger_webhook(unread)  # uncomment when API is ready